In [0]:
from pyspark.sql.types import *
import uuid

In [0]:
# Audit table name
AUDIT_TABLE = ("workspace.default.audit_pipeline_runs")

In [0]:
# Create run id
def get_run_id():
    return str(uuid.uuid4())

In [0]:
# Audit Schema
audit_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("input_count", LongType(), True),
    StructField("output_count", LongType(), True),
    StructField("error_message", StringType(), True)
])

In [0]:
# Create Audit Table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT_TABLE}
(
    run_id STRING,
    pipeline_name STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    status STRING,
    input_count BIGINT,
    output_count BIGINT,
    error_message STRING
)
USING DELTA
""")


DataFrame[]

In [0]:
# Reusable Audit Function
def write_audit_record(
    run_id,
    pipeline_name,
    start_time,
    end_time,
    status,
    input_count,
    output_count,
    error_message=None
):

    audit_data = [
    (
        str(run_id),
        pipeline_name,
        start_time,
        end_time,
        status,
        input_count,
        output_count,
        error_message
    )
    ]

    audit_df = spark.createDataFrame(
    audit_data,
    schema=audit_schema
    )
    logger.info("Writing logs to audit table")
    (
        audit_df.write
        .mode("append")
        .saveAsTable(
            AUDIT_TABLE
        )
    )

In [0]:
# # Test
# write_audit_record(
#     run_id=get_run_id(),
#     pipeline_name="test_pipeline",
#     start_time=datetime.now(),
#     end_time=datetime.now(),
#     status="SUCCESS",
#     input_count=100,
#     output_count=95,
#     error_message=None
# )

In [0]:
# spark.sql(f"""
# SELECT *
# FROM {AUDIT_TABLE}
# """).show(truncate=False)

+------------------------------------+-------------+--------------------------+--------------------------+-------+-----------+------------+--------------+
|run_id                              |pipeline_name|start_time                |end_time                  |status |input_count|output_count|error_message |
+------------------------------------+-------------+--------------------------+--------------------------+-------+-----------+------------+--------------+
|72a74f1d-8e83-4984-a841-a00ff284c99c|test_pipeline|2026-06-04 21:19:09.533181|2026-06-04 21:19:09.533184|FAILED |100        |0           |Sample Failure|
|ee472a0f-c491-48b3-9507-5e695be986cf|test_pipeline|2026-06-04 21:18:26.302722|2026-06-04 21:18:26.302726|SUCCESS|100        |95          |NULL          |
|47d48e60-9885-451c-a19e-e4601219508a|test_pipeline|2026-06-04 21:21:44.551147|2026-06-04 21:21:44.551151|SUCCESS|100        |95          |NULL          |
+------------------------------------+-------------+------------------

In [0]:
# # Sample filure record
# write_audit_record(
#     run_id=get_run_id(),
#     pipeline_name="test_pipeline",
#     start_time=datetime.now(),
#     end_time=datetime.now(),
#     status="FAILED",
#     input_count=100,
#     output_count=0,
#     error_message="Sample Failure"
# )

In [0]:
# spark.sql(f"""
# SELECT *
# FROM {AUDIT_TABLE}
# ORDER BY start_time DESC
# """).show(truncate=False)

+------------------------------------+----------------------+--------------------------+--------------------------+-------+-----------+------------+-------------+
|run_id                              |pipeline_name         |start_time                |end_time                  |status |input_count|output_count|error_message|
+------------------------------------+----------------------+--------------------------+--------------------------+-------+-----------+------------+-------------+
|5cefa9c7-f53a-4957-a3f9-0e4930131e7a|gold_daily_sales      |2026-06-04 22:15:41.177642|2026-06-04 22:15:51.753632|SUCCESS|531285     |305         |NULL         |
|5cefa9c7-f53a-4957-a3f9-0e4930131e7a|gold_customer_sales   |2026-06-04 22:15:41.177642|2026-06-04 22:15:51.753632|SUCCESS|531285     |4339        |NULL         |
|5cefa9c7-f53a-4957-a3f9-0e4930131e7a|gold_country_sales    |2026-06-04 22:15:41.177642|2026-06-04 22:15:51.753632|SUCCESS|531285     |38          |NULL         |
|968b8d33-9362-4bc7-af

In [0]:
# %sql
# SELECT *
# FROM workspace.default.audit_pipeline_runs

run_id,pipeline_name,start_time,end_time,status,input_count,output_count,error_message
968b8d33-9362-4bc7-afb8-c5451f767829,silver_online_pipeline,2026-06-04T22:14:47.112Z,2026-06-04T22:14:54.429Z,SUCCESS,541909,531285,null
755225e1-58c8-4bb5-8fdf-b0a9da8222f6,bronze_online_retail,2026-06-04T22:13:54.464Z,2026-06-04T22:14:04.398Z,SUCCESS,541909,541909,null
5cefa9c7-f53a-4957-a3f9-0e4930131e7a,gold_customer_sales,2026-06-04T22:15:41.177Z,2026-06-04T22:15:51.753Z,SUCCESS,531285,4339,null
5cefa9c7-f53a-4957-a3f9-0e4930131e7a,gold_country_sales,2026-06-04T22:15:41.177Z,2026-06-04T22:15:51.753Z,SUCCESS,531285,38,null
5cefa9c7-f53a-4957-a3f9-0e4930131e7a,gold_daily_sales,2026-06-04T22:15:41.177Z,2026-06-04T22:15:51.753Z,SUCCESS,531285,305,null
